# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports

import os
import requests
import ollama
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
# set up environment

In [ ]:
# here is the question; type over this to ask something new

question = """
Please explain the difference between overlap weighting and matched weighting in the context of causal inference.
"""

In [25]:
# prompts

system_prompt = "You are a helpful technical tutor who answers questions about causal inference and experimentation design"
user_prompt = "Please explain the difference between overlap weighting and matched weighting in the context of causal inference"

In [26]:
# messages

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

### Get gpt-4o-mini to answer, with streaming

In [20]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL_GPT = 'gpt-4o-mini'
openai = OpenAI()

API key looks good so far


In [27]:
stream = openai.chat.completions.create(model=MODEL_GPT, messages=messages,stream=True)
    
response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    response = response.replace("```","").replace("markdown", "")
    update_display(Markdown(response), display_id=display_handle.display_id)

Overlap weighting and matched weighting are both methods used in causal inference to adjust for confounding and to better estimate treatment effects, particularly in observational studies where random assignment is not possible. While their goals are similar, they differ in their approach to balancing the treatment groups.

### Overlap Weighting

**Overlap weighting** is a method that emphasizes the importance of common support between treated and control groups. The basic idea is to assign weights to individuals in different groups based on the estimated propensity score, which is the probability of receiving treatment given observed covariates.

Key features of overlap weighting:
1. **Weight Calculation**: In overlap weighting, individuals are assigned weights that reflect how similar they are in terms of their propensity scores. Specifically, individuals in the treatment group are weighted by the inverse of their propensity score, and those in the control group are weighted by the inverse of one minus their propensity score.
   
   \[
   w_i = \frac{P(T=1 | X)}{\text{min}(P(T=1 | X), P(T=0 | X))}
   \]

2. **Focus on Common Support**: By using weights based on propensity scores, overlap weighting helps ensure that individuals with similar characteristics across treatment and control groups are given more emphasis, thus improving the balance between the groups.

3. **Managing Imbalance**: The aim is to create a pseudo-population in which the characteristics of the treated and control groups are more balanced. This can improve the validity of causal inferences.

### Matched Weighting

**Matched weighting**, on the other hand, typically refers to the procedure of matching treated individuals with control individuals who have similar characteristics based on covariates, often facilitated by propensity scores. After matching, weights are assigned based on the matched pairs.

Key features of matched weighting:
1. **Matching Process**: In matched weighting, individuals are paired based on their similarity (e.g., using propensity scores) and thus matched treated and control units are compared directly to estimate treatment effects.
  
2. **Weighting in Matched Samples**: After pairing treated and control individuals, the weights may be set to adjust for the degree of matches; for instance, each treated individual could be given a weight proportionate to the number of matched controls.

3. **Selection Bias Mitigation**: The intent is to create a control group that resembles the treatment group as closely as possible and to mitigate selection bias by directly controlling for covariates rather than weighting based on propensity scores.

### Summary of Differences

- **Methodology**: Overlap weighting uses a continuous adjustment based on the propensity score across the entire sample, while matched weighting focuses on creating pairs of treated and control units based on their similarities.
  
- **Application**: Overlap weighting applies weights based on the degree of overlap in propensity scores, while matched weighting relies heavily on the pairings of individuals, possibly leading to a smaller sample after matching.

- **Goal**: Both aim to better estimate causal effects by balancing the treatment and control groups but do so via different mechanisms — overlap weighting through statistical adjustment and matched weighting through direct pairwise comparisons.

Choosing between these methods depends on the context of the study, the data available, and specific research questions of interest.

#### Alternative Approach for OpenAI

In [24]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

messages = [
    {"role": "user", "content": "Please explain the difference between overlap weighting and matched weighting in the context of causal inference"}
]
response = openai.chat.completions.create(model=MODEL_GPT, messages=messages, stream=False)
answer = print(response.choices[0].message.content)
answer

In the context of causal inference, both overlap weighting and matched weighting are techniques used to balance covariates between treatment and control groups in observational studies to improve causal estimates. However, they do so using different methodologies and have distinct characteristics.

### Overlap Weighting

Overlap weighting is a method designed to address the issue of common support in observational studies. It focuses on ensuring that individuals in the treatment and control groups have similar covariate distributions. The main idea behind overlap weighting is to give more weight to individuals in regions of the covariate space where there is overlap between the treatment and control groups, while down-weighting individuals in regions where there is no overlap.

1. **Mechanism**: Overlap weighting typically involves using the inverse probability of treatment weighting (IPTW) approach by calculating weights based on the estimated propensity scores for receiving treatment

### Get Llama 3.2 to answer

In [ ]:
# Get Llama 3.2 to answer
# activated Ollama first by: ollama serve (first bash terminal)
# second by: ollama pull llama3.2 (second bash terminal)

response = ollama.chat(model=MODEL_LLAMA, messages=messages)
reply = response['message']['content']
display(Markdown(reply))

In the context of causal inference, both overlap weighting and matched weighting are techniques used to address the problem of unidentifiability of treatment effects when there is no overlap between treated and control groups. Here's a brief overview of each:

**Overlap Weighting**

Overlap weighting is a technique developed by Angrist and Pischke (2008) that uses overlap to estimate causal effects. The idea is to assign weights to observations in both the treated and control groups, based on how much they are likely to be similar if they were matched with each other.

Here's how it works:

1. Estimate a baseline estimate of the treatment effect using an instrumental variable (IV).
2. Calculate the overlap between treated and control units using the IV.
3. Assign weights to observations in both groups, based on their overlap. The weight is calculated as the ratio of the probability that a treated unit would be matched with a control unit, given the observed covariates.
4. Use these weights to calculate an estimate of the treatment effect.

The idea behind overlap weighting is that by using overlap to "weight" observations in both groups, we can effectively use all available data to estimate causal effects, even when there's no overlap between treated and control units.

**Matched Weighting**

Matched weighting is another technique developed by Wooldridge (2008) that uses matching to estimate causal effects. The idea is to match treated and control units on observed covariates, rather than using IVs.

Here's how it works:

1. Estimate a baseline model of the treatment effect using only the observed covariates.
2. Match treated and control units based on these covariates.
3. Use the matched pairs to estimate the treatment effect.

The idea behind matched weighting is that by matching treated and control units, we can eliminate many of the sources of bias in the data and get a more accurate estimate of the causal effect.

**Key differences**

Here are the key differences between overlap weighting and matched weighting:

* **Instrumental variable vs. observed covariates**: Overlap weighting uses IVs to estimate the baseline treatment effect, while matched weighting uses only observed covariates.
* **Weighting vs. matching**: Overlap weighting assigns weights to observations in both groups, while matched weighting matches treated and control units based on observed covariates.
* **Assumptions**: Matched weighting requires that the data be balanced across all covariates, while overlap weighting makes no such assumption.

In general, overlap weighting is considered more flexible than matched weighting, as it allows for the use of IVs to estimate the baseline treatment effect. However, both techniques can be used to address unidentifiability in causal inference, and the choice between them will depend on the specific research question and data characteristics.

#### Alternative Approach for Ollama

In [8]:
# Constants

OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {"Content-Type": "application/json"}
MODEL_LLAMA = "llama3.2"

In [9]:
# Create a messages list using the same format that we used for OpenAI

messages = [
    {"role": "user", "content": "Please explain the difference between overlap weighting and matched weighting in the context of causal inference"}
]

In [16]:
payload = {
        "model": MODEL_LLAMA,
        "messages": messages,
        "stream": False
    }

In [ ]:
# Let's just make sure the model is loaded
# activated Ollama first by: ollama serve (first bash terminal)
# second by: ollama pull llama3.2 (second bash terminal)

!ollama pull llama3.2

pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [18]:
# answering the question using Llama 3.2
response = requests.post(OLLAMA_API, json=payload, headers=HEADERS)
print(response.json()['message']['content'])

In the context of causal inference, overlap weighting and matched weighting are two techniques used to estimate treatment effects in observational studies where there is no randomization.

**Overlap Weighting**

Overlap weighting is a method for estimating treatment effects when the data consists of independent observations. It assumes that each unit (e.g., individual or firm) receives a different treatment assignment, and that the observed covariates are independent of both the treatment assignment and the treatment effect.

The basic idea behind overlap weighting is to assign weights to each observation based on its "overlap" with other observations in the same treatment group. Specifically, for an observation i in treatment group T, the weight is calculated as:

w_i = n_T / (n_T - k)

where n_T is the number of observations in treatment group T, and k is a parameter that controls the strength of overlap.

The idea behind this weighting scheme is to "normalize" each observation by it

#### Using openai package to call ollama 3,2

In [19]:
import ollama

response = ollama.chat(model=MODEL_LLAMA, messages=messages)
print(response['message']['content'])

In the context of causal inference, overlap weighting and matched weighting are two methods used to address the issue of confounding in observational data. The main goal is to estimate the effect of a treatment or intervention on an outcome variable while controlling for potential confounding variables.

**Matched Weighting**

Matched weighting is a method that aims to balance the distribution of covariates between treated and control groups by matching individuals with similar observed covariate values (e.g., age, sex, income). The idea behind matched weighting is that if the confounders are well-matched, the differences in outcomes between treatment and control groups will be largely due to the effect of the treatment itself.

The process involves:

1. Identifying potential confounding variables (e.g., covariates)
2. Calculating the weights for each individual based on their observed values for these confounders
3. Applying these weights to the outcome variable, so that individuals w